In [113]:
import pandas as pd
import sqlite3

In [114]:

# Load your cleaned weather data
weather_df = pd.read_csv("../cleaned_climate_data.csv")

# Load the SimpleMaps cities dataset
simple_map_cities_df = pd.read_csv("../worldcities.csv")

In [115]:
weather_df.head()

,city,region,country,month,high_temp_F,low_temp_F,mean_temp_F,precipitation_in,humidity_percent,dew_point_F,wind_mph,pressure_Hg,visibility_mi
0,Accra,NaN,Ghana,January,89.0,75.0,82.0,1.57,75.0,73.0,14.0,29.85,5.0
1,Accra,NaN,Ghana,February,90.0,77.0,84.0,1.79,77.0,75.0,17.0,29.84,6.0
2,Accra,NaN,Ghana,March,91.0,78.0,84.0,2.29,78.0,76.0,17.0,29.83,7.0
3,Accra,NaN,Ghana,April,90.0,78.0,84.0,2.93,78.0,76.0,16.0,29.83,7.0
4,Accra,NaN,Ghana,May,89.0,77.0,83.0,5.11,80.0,75.0,15.0,29.87,8.0


In [116]:
simple_map_cities_df.head()

,city,city_ascii,lat,lng,country,iso2,iso3,admin_name,capital,population,id
0,Tokyo,Tokyo,35.6850,139.7514,Japan,JP,JPN,Tōkyō,primary,39105000.0,1.392686e+09
1,Jakarta,Jakarta,-6.1753,106.8269,Indonesia,ID,IDN,Jakarta,primary,33756000.0,1.360771e+09
2,Guangzhou,Guangzhou,23.1300,113.2600,China,CN,CHN,Guangdong,admin,26940000.0,1.156237e+09
3,Mumbai,Mumbai,19.0758,72.8775,India,IN,IND,Mahārāshtra,admin,24973000.0,1.356227e+09
4,Shanghai,Shanghai,31.2325,121.4692,China,CN,CHN,Shanghai,admin,24870895.0,1.156074e+09


In [117]:
simple_map_countries = simple_map_cities_df['country'].unique()
print(f"Total number of contries in simple map df: {len(simple_map_countries)}")

Total number of contries in simple map df: 242


In [118]:
weather_countries = weather_df['country'].unique()
print(f"Total number of contries in weather df: {len(weather_countries)}")

Total number of contries in weather df: 92


In [119]:
diff = set(weather_countries).difference(set(simple_map_countries))
print(diff)

{'Myanmar', 'Bahamas', 'USA', 'South Korea', 'Congo Dem. Rep.'}


In [120]:
# A list of keywords to search for in the SimpleMaps country column
search_terms = [
    'Congo', 
    'Bahama', 
    'Kazak', 
    'Czech', 
    'Kore', 
    'United',  # For USA and UK
    'Myanmar', 
    'Burma'    # Sometimes Myanmar is listed as Burma
]

print("--- SimpleMaps Country Matches ---")
for term in search_terms:
    # Find all unique country names that contain the search term
    matches = simple_map_cities_df[
        simple_map_cities_df['country'].str.contains(term, case=False, na=False)
    ]['country'].unique()
    
    print(f"Searching '{term}': {matches}")

--- SimpleMaps Country Matches ---
Searching 'Congo': ['Congo (Kinshasa)' 'Congo (Brazzaville)']
Searching 'Bahama': ['Bahamas, The']
Searching 'Kazak': ['Kazakhstan']
Searching 'Czech': ['Czechia']
Searching 'Kore': ['Korea, South' 'Korea, North']
Searching 'United': ['United States' 'United Kingdom' 'United Arab Emirates']
Searching 'Myanmar': []
Searching 'Burma': ['Burma']


In [121]:
country_corrections = {
    "Congo Dem. Rep.": "Congo (Kinshasa)",
    "Bahamas": "Bahamas, The",
    "Kazakstan": "Kazakhstan",
    "Czech Republic": "Czechia",
    "South Korea": "Korea, South",
    "USA": "United States",
    "UK": "United Kingdom",
    "Myanmar": "Burma"
}

# Apply the corrections to scraped weather data
weather_df['country'] = weather_df['country'].replace(country_corrections)

In [122]:
simple_map_cities_df = simple_map_cities_df.rename(columns={"admin_name": "region"})

In [123]:
def create_join_keys(df, columns):
    """
    Creates temporary '_clean' columns for merging by stripping accents, 
    whitespace.
    """
    # Work on a copy to avoid SettingWithCopy warnings
    df_clean = df.copy()
    
    for col in columns:
        clean_col_name = f"{col}_clean"
        df_clean[clean_col_name] = (
            df_clean[col]
            .str.normalize('NFKD')               # Decompose accented characters
            .str.encode('ascii', errors='ignore') # Drop the accents
            .str.decode('utf-8')                 # Convert back to standard string
            .str.strip()                         # Remove leading/trailing spaces
        )
    return df_clean

In [124]:
geo_columns = ['city', 'region', 'country']

# Generate the clean join keys for both datasets
weather_df = create_join_keys(weather_df, geo_columns)
simple_map_cities_df = create_join_keys(simple_map_cities_df, geo_columns)

In [125]:
simple_map_countries = simple_map_cities_df['country_clean'].unique()
print(f"Total number of contries in simple map df: {len(simple_map_countries)}")

weather_countries = weather_df['country_clean'].unique()
print(f"Total number of contries in weather df: {len(weather_countries)}")

diff = set(weather_countries).difference(set(simple_map_countries))
print(f"Countries in weather df but not in simple map df: {diff}")


Total number of contries in simple map df: 242
Total number of contries in weather df: 92
Countries in weather df but not in simple map df: set()


In [126]:
simple_map_cities = simple_map_cities_df['city_clean'].unique()
print(f"Total number of cities in simple map df: {len(simple_map_cities)}")

Total number of cities in simple map df: 46256


In [127]:
weather_cities = weather_df['city_clean'].unique()
print(f"Total number of cities in weather df: {len(weather_cities)}")

Total number of cities in weather df: 140


In [128]:
diff = set(weather_cities).difference(set(simple_map_cities))
print(diff)

{'Kiritimati', 'Washington DC', 'Bengaluru', 'Yangon'}


In [129]:
# A list of partial names to search for in the SimpleMaps city column
search_cities = [
    'Yangon', 'Rangoon',      # Checking both names for Myanmar's largest city
    'Kiritimati', 'Christmas',# Kiritimati is also known as Christmas Island
    'Bengaluru', 'Bangalore', # Checking both names for the Indian city
    'Washington'              # Checking Washington DC
]

print("--- SimpleMaps City Matches ---")
for term in search_cities:
    # Find all unique city names that contain the search term
    matches = simple_map_cities_df[
        simple_map_cities_df['city_clean'].str.contains(term, case=False, na=False)
    ]['city_clean'].unique()
    
    # We only print if it actually found a match to keep the output clean
    if len(matches) > 0:
        print(f"Searching '{term}': {matches}")

--- SimpleMaps City Matches ---
Searching 'Rangoon': ['Rangoon']
Searching 'Bangalore': ['Bangalore']
Searching 'Washington': ['Washington' 'New Washington' 'Fort Washington' 'Mount Washington'
 'Port Washington' 'Washington Court House' 'Washington Terrace']


In [130]:
city_corrections = {
    "Washington DC": "Washington",
    "Bengaluru": "Bangalore",
    "Yangon": "Rangoon" 
}

# Apply the corrections to your scraped weather data
weather_df['city_clean'] = weather_df['city_clean'].replace(city_corrections)

In [131]:
simple_map_cities = simple_map_cities_df['city_clean'].unique()
print(f"Total number of cities in simple map df: {len(simple_map_cities)}")

weather_cities = weather_df['city_clean'].unique()
print(f"Total number of cities in weather df: {len(weather_cities)}")

diff = set(weather_cities).difference(set(simple_map_cities))
print(f"Cities in weather df but not in simple map df: {diff}")

Total number of cities in simple map df: 46256
Total number of cities in weather df: 140
Cities in weather df but not in simple map df: {'Kiritimati'}


In [132]:
# Create sets of tuples (city, region, country) for both datasets
weather_set = set(weather_df[['city_clean', 'region_clean', 'country_clean']].itertuples(index=False, name=None))
map_set = set(simple_map_cities_df[['city_clean', 'region_clean', 'country_clean']].itertuples(index=False, name=None))

# Subtract the map set from the weather set to see what is leftover
missing_in_maps = weather_set - map_set
print(missing_in_maps)

{('Sofia', nan, 'Bulgaria'), ('Washington', nan, 'United States'), ('Bogota', nan, 'Colombia'), ('Hanoi', nan, 'Vietnam'), ('Tokyo', nan, 'Japan'), ('Athens', nan, 'Greece'), ('Kinshasa', nan, 'Congo (Kinshasa)'), ('Havana', nan, 'Cuba'), ('Budapest', nan, 'Hungary'), ('Kuala Lumpur', nan, 'Malaysia'), ('Copenhagen', nan, 'Denmark'), ('Beirut', nan, 'Lebanon'), ('La Paz', nan, 'Bolivia'), ('Oslo', nan, 'Norway'), ('Prague', nan, 'Czechia'), ('Rangoon', nan, 'Burma'), ('Shanghai', 'Shanghai Municipality', 'China'), ('Amsterdam', nan, 'Netherlands'), ('Kiritimati', 'Christmas Island', 'Kiribati'), ('Zagreb', nan, 'Croatia'), ('Amman', nan, 'Jordan'), ('Auckland', nan, 'New Zealand'), ('Tallinn', nan, 'Estonia'), ('Dublin', nan, 'Ireland'), ('Johannesburg', nan, 'South Africa'), ('Harare', nan, 'Zimbabwe'), ('Almaty', nan, 'Kazakhstan'), ('Manila', nan, 'Philippines'), ('Islamabad', nan, 'Pakistan'), ('Kyiv', nan, 'Ukraine'), ('Istanbul', nan, 'Turkey'), ('Guatemala City', nan, 'Guatemala

In [133]:
# Create a lookup table from SimpleMaps
# Sort by population to ensure we grab the region for the major city, not a tiny duplicate
map_lookup = (
    simple_map_cities_df.sort_values('population', ascending=False)
    .drop_duplicates(subset=['city_clean', 'country_clean'])
)
map_lookup.head()

,city,city_ascii,lat,lng,country,iso2,iso3,region,capital,population,id,city_clean,region_clean,country_clean
0,Tokyo,Tokyo,35.6850,139.7514,Japan,JP,JPN,Tōkyō,primary,39105000.0,1.392686e+09,Tokyo,Tokyo,Japan
1,Jakarta,Jakarta,-6.1753,106.8269,Indonesia,ID,IDN,Jakarta,primary,33756000.0,1.360771e+09,Jakarta,Jakarta,Indonesia
2,Guangzhou,Guangzhou,23.1300,113.2600,China,CN,CHN,Guangdong,admin,26940000.0,1.156237e+09,Guangzhou,Guangdong,China
3,Mumbai,Mumbai,19.0758,72.8775,India,IN,IND,Mahārāshtra,admin,24973000.0,1.356227e+09,Mumbai,Maharashtra,India
4,Shanghai,Shanghai,31.2325,121.4692,China,CN,CHN,Shanghai,admin,24870895.0,1.156074e+09,Shanghai,Shanghai,China


In [134]:
# Merge the region column into weather data
weather_df = pd.merge(
    weather_df,
    map_lookup[["city_clean", "country_clean", "region_clean"]],
    on=["city_clean", "country_clean"],
    how="left",
    suffixes=('', "_from_map")
)
weather_df.head()

,city,region,country,month,high_temp_F,low_temp_F,mean_temp_F,precipitation_in,humidity_percent,dew_point_F,wind_mph,pressure_Hg,visibility_mi,city_clean,region_clean,country_clean,region_clean_from_map
0,Accra,NaN,Ghana,January,89.0,75.0,82.0,1.57,75.0,73.0,14.0,29.85,5.0,Accra,NaN,Ghana,Greater Accra
1,Accra,NaN,Ghana,February,90.0,77.0,84.0,1.79,77.0,75.0,17.0,29.84,6.0,Accra,NaN,Ghana,Greater Accra
2,Accra,NaN,Ghana,March,91.0,78.0,84.0,2.29,78.0,76.0,17.0,29.83,7.0,Accra,NaN,Ghana,Greater Accra
3,Accra,NaN,Ghana,April,90.0,78.0,84.0,2.93,78.0,76.0,16.0,29.83,7.0,Accra,NaN,Ghana,Greater Accra
4,Accra,NaN,Ghana,May,89.0,77.0,83.0,5.11,80.0,75.0,15.0,29.87,8.0,Accra,NaN,Ghana,Greater Accra


In [135]:
weather_df['region_clean'] = weather_df['region_clean'].fillna(weather_df['region_clean_from_map'])
weather_df = weather_df.drop(columns=['region_clean_from_map'])
weather_df.head()

,city,region,country,month,high_temp_F,low_temp_F,mean_temp_F,precipitation_in,humidity_percent,dew_point_F,wind_mph,pressure_Hg,visibility_mi,city_clean,region_clean,country_clean
0,Accra,NaN,Ghana,January,89.0,75.0,82.0,1.57,75.0,73.0,14.0,29.85,5.0,Accra,Greater Accra,Ghana
1,Accra,NaN,Ghana,February,90.0,77.0,84.0,1.79,77.0,75.0,17.0,29.84,6.0,Accra,Greater Accra,Ghana
2,Accra,NaN,Ghana,March,91.0,78.0,84.0,2.29,78.0,76.0,17.0,29.83,7.0,Accra,Greater Accra,Ghana
3,Accra,NaN,Ghana,April,90.0,78.0,84.0,2.93,78.0,76.0,16.0,29.83,7.0,Accra,Greater Accra,Ghana
4,Accra,NaN,Ghana,May,89.0,77.0,83.0,5.11,80.0,75.0,15.0,29.87,8.0,Accra,Greater Accra,Ghana


In [136]:
# Create sets of tuples (city, region, country) for both datasets
weather_set = set(weather_df[['city_clean', 'region_clean', 'country_clean']].itertuples(index=False, name=None))
map_set = set(simple_map_cities_df[['city_clean', 'region_clean', 'country_clean']].itertuples(index=False, name=None))

# Subtract the map set from the weather set to see what is leftover
missing_in_maps = weather_set - map_set
stubborn_cities = [item[0] for item in missing_in_maps]
print(stubborn_cities)

['Jakarta', 'London', 'Barcelona', 'Vienna', 'Paris', 'Dubai', 'Shanghai', 'Beijing', 'Kiritimati']


In [137]:
import numpy as np

# Force their region to NaN in the weather dataframe
weather_df.loc[weather_df['city_clean'].isin(stubborn_cities), 'region_clean'] = np.nan

# Now run the backfill lookup
map_lookup = (
    simple_map_cities_df.sort_values('population', ascending=False)
    .drop_duplicates(subset=['city_clean', 'country_clean'])
)

weather_df = pd.merge(
    weather_df,
    map_lookup[['city_clean', 'country_clean', 'region_clean']], 
    on=['city_clean', 'country_clean'],
    how='left',
    suffixes=('', '_from_map') 
)

# Because those 9 cities are now NaN, this will overwrite them with the perfect SimpleMaps region
weather_df['region_clean'] = weather_df['region_clean'].fillna(weather_df['region_clean_from_map'])
weather_df = weather_df.drop(columns=['region_clean_from_map'])

In [138]:
# Create sets of tuples (city, region, country) for both datasets
weather_set = set(weather_df[['city_clean', 'region_clean', 'country_clean']].itertuples(index=False, name=None))
map_set = set(simple_map_cities_df[['city_clean', 'region_clean', 'country_clean']].itertuples(index=False, name=None))

# Subtract the map set from the weather set to see what is leftover
missing_in_maps = weather_set - map_set
print(missing_in_maps)

{('Kiritimati', nan, 'Kiribati')}


In [139]:
weather_df = weather_df[~((weather_df['city'] == 'Kiritimati') & (weather_df['country'] == 'Kiribati'))]
weather_df = weather_df.reset_index(drop=True)

In [140]:
cities_subset = simple_map_cities_df[['city_clean', 'country_clean', 'region_clean', 'population', 'lat', 'lng']]

# Sort by population and drop duplicates so the merge keys are perfectly unique
cities_subset = (
    cities_subset
    .sort_values('population', ascending=False)
    .drop_duplicates(subset=['city_clean', 'region_clean', 'country_clean'])
)

# Merge two dataframes
final_df = pd.merge(
    weather_df, 
    cities_subset, 
    on=['city_clean', 'region_clean', 'country_clean'],
    how='left',
    validate='many_to_one'
)

final_df.head()

,city,region,country,month,high_temp_F,low_temp_F,mean_temp_F,precipitation_in,humidity_percent,dew_point_F,wind_mph,pressure_Hg,visibility_mi,city_clean,region_clean,country_clean,population,lat,lng
0,Accra,NaN,Ghana,January,89.0,75.0,82.0,1.57,75.0,73.0,14.0,29.85,5.0,Accra,Greater Accra,Ghana,1782150.0,5.556,-0.1969
1,Accra,NaN,Ghana,February,90.0,77.0,84.0,1.79,77.0,75.0,17.0,29.84,6.0,Accra,Greater Accra,Ghana,1782150.0,5.556,-0.1969
2,Accra,NaN,Ghana,March,91.0,78.0,84.0,2.29,78.0,76.0,17.0,29.83,7.0,Accra,Greater Accra,Ghana,1782150.0,5.556,-0.1969
3,Accra,NaN,Ghana,April,90.0,78.0,84.0,2.93,78.0,76.0,16.0,29.83,7.0,Accra,Greater Accra,Ghana,1782150.0,5.556,-0.1969
4,Accra,NaN,Ghana,May,89.0,77.0,83.0,5.11,80.0,75.0,15.0,29.87,8.0,Accra,Greater Accra,Ghana,1782150.0,5.556,-0.1969


In [141]:
final_df = final_df.drop(columns=["city", "region", "country"])

final_df = final_df.rename(columns={
    "city_clean": "city",
    "region_clean": "region",
    "country_clean": "country"
})

In [142]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1668 entries, 0 to 1667
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   month             1668 non-null   object 
 1   high_temp_F       1668 non-null   float64
 2   low_temp_F        1668 non-null   float64
 3   mean_temp_F       1668 non-null   float64
 4   precipitation_in  1668 non-null   float64
 5   humidity_percent  1668 non-null   float64
 6   dew_point_F       1668 non-null   float64
 7   wind_mph          1668 non-null   float64
 8   pressure_Hg       1668 non-null   float64
 9   visibility_mi     1656 non-null   float64
 10  city              1668 non-null   object 
 11  region            1620 non-null   object 
 12  country           1668 non-null   object 
 13  population        1668 non-null   float64
 14  lat               1668 non-null   float64
 15  lng               1668 non-null   float64
dtypes: float64(12), object(4)
memory usage: 20

In [143]:
row_counts = final_df.groupby(["city", "country"]).size().reset_index(name="row_count")

anomalies = row_counts[row_counts["row_count"] != 12]
anomalies

,city,country,row_count


In [144]:
# Save your final enriched dataset
final_df.to_csv("../climate_and_population_data.csv", index=False)

In [145]:
# Open connection to SQLite database
conn = sqlite3.connect("../db/city_climate_data.db")

# Save the final DataFrame to a SQLite database
final_df.to_sql("city_climate_data", conn, if_exists='replace', index=False)

# Quick verification query to confirm the write is succeeded
row_count = conn.execute("SELECT COUNT(*) FROM city_climate_data").fetchone()[0]
print(f"Successfully saved {row_count:,} rows to city_climate_data table.")

# Close the connection
conn.close()

Successfully saved 1,668 rows to city_climate_data table.
